In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, MultiHeadAttention, LayerNormalization, Input, GlobalAveragePooling1D
from tensorflow.keras import Model

FILE_NAME ='/content/drive/MyDrive/thesis_ready_data.csv'
INPUT_WINDOW = 60
PREDICT_WINDOW = 12
BATCH_SIZE = 32
EPOCHS = 50

print("Loading data...")
df = pd.read_csv(FILE_NAME)

df = df.fillna(0)

features = ['cpu', 'rps', 'latency_p95']
target = ['cpu']

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df[features])

X, y = [], []
for i in range(len(data_scaled) - INPUT_WINDOW - PREDICT_WINDOW):
    X.append(data_scaled[i : i + INPUT_WINDOW])
    y.append(data_scaled[i + INPUT_WINDOW : i + INPUT_WINDOW + PREDICT_WINDOW, 0])

X = np.array(X)
y = np.array(y)

split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Training Shape: {X_train.shape}")
print(f"Testing Shape:  {X_test.shape}")

def build_transformer_model(input_shape, head_size=256, num_heads=4, ff_dim=4, dropout=0.1):
    inputs = Input(shape=input_shape)

    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x)
    x = x + inputs

    x = GlobalAveragePooling1D()(x)
    x = Dropout(dropout)(x)
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)

    outputs = Dense(PREDICT_WINDOW, activation="linear")(x)

    return Model(inputs=inputs, outputs=outputs)

model = build_transformer_model(input_shape=(INPUT_WINDOW, len(features)))
model.compile(optimizer='adam', loss='mse')

print("Starting Training...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Training Progress')
plt.ylabel('Error (MSE)')
plt.xlabel('Epoch')
plt.legend()
plt.show()

model.save("thesis_model.keras")
print("Model saved as 'thesis_model.keras'")

Loading data...
Training Shape: (3224, 60, 3)
Testing Shape:  (807, 60, 3)
Starting Training...
Epoch 1/50
101/101 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - loss: 0.4207